# NavDrishti causal baselines (CPU)
Choose exactly one source route. The ZIP route supports unpushed local work and never changes main.

In [ ]:
import os, sys, subprocess, pathlib
assert sys.version_info >= (3, 9)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'pyyaml', 'pytest'], check=True)
print('CPU preflight OK', sys.version)


In [ ]:
# ZIP route: upload causal_baseline_source.zip, then point SOURCE_ZIP at it.
SOURCE_ZIP = '/content/causal_baseline_source.zip'
REPO_URL, REVISION = '', ''
root = pathlib.Path('/content/NavDrishti')
if pathlib.Path(SOURCE_ZIP).exists():
    import zipfile
    root.mkdir(exist_ok=True)
    zipfile.ZipFile(SOURCE_ZIP).extractall(root)
elif REPO_URL and REVISION:
    subprocess.run(['git','clone',REPO_URL,str(root)], check=True)
    subprocess.run(['git','-C',str(root),'checkout','--detach',REVISION], check=True)
else:
    raise RuntimeError('Provide a source ZIP or repository URL plus immutable revision')
os.chdir(root)


In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'experiments/causal_baseline/tests', '-q'], text=True)
if r.returncode:
    raise RuntimeError('tests failed or zero collection')


In [ ]:
# Upload one verified, materialized PHONE CSV from a documented training session; no vehicle file is used.
PHONE_CSV = '/content/S-training-session.csv'
if not pathlib.Path(PHONE_CSV).is_file():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1: raise RuntimeError('upload exactly one phone CSV')
    PHONE_CSV = '/content/' + next(iter(uploaded))
if pathlib.Path(PHONE_CSV).read_bytes()[:64].startswith(b'version https://git-lfs.github.com'): raise RuntimeError('LFS pointer, not CSV')
import json, shutil
cfg = json.loads(pathlib.Path('experiments/causal_baseline/configs/example.json').read_text())
cfg.update({'phone_csv': PHONE_CSV, 'session': 'documented-training-session'})
result_root = pathlib.Path('results/causal_baseline_colab'); result_root.mkdir(parents=True, exist_ok=True)
for mode in ('constant_velocity', 'gyro_heading_speed'):
    arm = dict(cfg, mode=mode)
    # Gyro mode deliberately has no mount matrix unless the uploaded session supplies a justified one.
    arm.pop('phone_to_vehicle_rotation', None)
    path = result_root / (mode + '.json'); path.write_text(json.dumps(arm, indent=2))
    subprocess.run([sys.executable, '-m', 'experiments.causal_baseline.run', '--config', str(path), '--output', str(result_root / mode)], check=True)
shutil.make_archive('/content/causal_baseline_results', 'zip', result_root)
print('results:', '/content/causal_baseline_results.zip')
